# **Word2Vec: обучение и анализ эмбеддингов**

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

**1. Подготовка данных и предобработка**

In [ ]:
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('stopwords')

# Чтение корпуса (каждая строка – отдельный документ)
def read_corpus(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        texts = f.readlines()
    return texts

corpus = read_corpus('corpus_news.txt')   # основной корпус – новости
print('Всего документов:', len(corpus))

stop_words = set(stopwords.words('russian'))

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)   # убираем пунктуацию
    text = re.sub(r'\d+', '', text)       # убираем цифры
    tokens = word_tokenize(text, language='russian')
    # удаляем стоп-слова и слишком короткие токены
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Предобработка всего корпуса
sentences = [clean_and_tokenize(doc) for doc in corpus if doc.strip()]
sentences = [s for s in sentences if len(s) > 0]
print('Предложений после очистки:', len(sentences))

**2. Обучение базовой модели (Задания 1–3)**

In [ ]:
# Модель по умолчанию: вектор 100, окно 5, min_count=2
model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)
model.save("word2vec_news.model")

# Задание 1: похожие слова
word = "россия"
if word in model.wv:
    similar = model.wv.most_similar(word, topn=5)
    print(f"5 слов, похожих на '{word}':")
    for w, score in similar:
        print(f"  {w}: {score:.4f}")

# Задание 2: аналогии
def analogy(positive, negative, model, topn=1):
    res = model.wv.most_similar(positive=positive, negative=negative, topn=topn)
    return res[0][0] if topn == 1 else res

print("король - мужчина + женщина ≈", analogy(['король', 'женщина'], ['мужчина'], model))
print("Paris - France + Germany ≈", analogy(['paris', 'germany'], ['france'], model))
print("good - bad + happy ≈", analogy(['good', 'happy'], ['bad'], model))

# Задание 3: косинусное сходство
def cosine_sim(word1, word2, model):
    try:
        return model.wv.similarity(word1, word2)
    except KeyError:
        return None

print("Близкие слова (автомобиль, машина):", cosine_sim("автомобиль", "машина", model))
print("Случайные слова (солнце, стол):", cosine_sim("солнце", "стол", model))

**3. Влияние гиперпараметров (Задание 4)**

In [ ]:
params = [
    {'window': 2, 'size': 50}, {'window': 2, 'size': 100}, {'window': 2, 'size': 300},
    {'window': 5, 'size': 50}, {'window': 5, 'size': 100}, {'window': 5, 'size': 300},
    {'window': 10, 'size': 50}, {'window': 10, 'size': 100}, {'window': 10, 'size': 300},
]

models_params = {}
for p in params:
    key = f"w{p['window']}_s{p['size']}"
    models_params[key] = Word2Vec(sentences, vector_size=p['size'], window=p['window'], 
                                  min_count=2, workers=4)
    print(f"Обучил {key}")

# Тест аналогии для сравнения
positive = ['король', 'женщина']
negative = ['мужчина']
for key, m in models_params.items():
    try:
        res = m.wv.most_similar(positive=positive, negative=negative, topn=1)[0][0]
        print(f"{key}: король - мужчина + женщина ≈ {res}")
    except:
        print(f"{key}: ошибка")

**4. CBOW vs Skip-gram (Задание 5)**

In [ ]:
model_cbow = Word2Vec(sentences, sg=0, vector_size=100, window=5, min_count=2)
model_sg   = Word2Vec(sentences, sg=1, vector_size=100, window=5, min_count=2)

freq_word = "россия"
rare_word = "эпистемология"   # редкое слово, если его нет в корпусе – заменить

print("CBOW, частое слово:", model_cbow.wv.most_similar(freq_word, topn=3))
print("Skip-gram, частое слово:", model_sg.wv.most_similar(freq_word, topn=3))
if rare_word in model_cbow.wv:
    print("CBOW, редкое слово:", model_cbow.wv.most_similar(rare_word, topn=3))
    print("Skip-gram, редкое слово:", model_sg.wv.most_similar(rare_word, topn=3))
else:
    print("Редкое слово отсутствует в словаре, обученном на новостях.")

**5. Визуализация эмбеддингов (Задание 6)**

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def plot_embeddings(model, method='pca', n_words=80):
    # Берём самые частотные слова
    words = [w for w, _ in model.wv.most_similar(positive=[], topn=n_words)]
    vectors = np.array([model.wv[w] for w in words])
    
    if method == 'pca':
        reducer = PCA(n_components=2)
    else:
        reducer = TSNE(n_components=2, random_state=42)
    
    coords = reducer.fit_transform(vectors)
    plt.figure(figsize=(12, 10))
    plt.scatter(coords[:, 0], coords[:, 1])
    for i, word in enumerate(words):
        plt.annotate(word, xy=(coords[i, 0], coords[i, 1]), fontsize=8)
    plt.title(f'Визуализация эмбеддингов ({method.upper()})')
    plt.show()

plot_embeddings(model, method='pca')
# plot_embeddings(model, method='tsne')   # t-SNE работает дольше

**6. Обучение на разных корпусах (Задание 7)**

In [ ]:
def train_corpus(file_path, model_name):
    texts = read_corpus(file_path)
    sents = [clean_and_tokenize(t) for t in texts if t.strip()]
    sents = [s for s in sents if len(s) > 0]
    m = Word2Vec(sents, vector_size=100, window=5, min_count=2)
    m.save(f"{model_name}.model")
    return m

models_corpus = {
    'news': train_corpus('news.txt', 'news'),
    'science': train_corpus('science.txt', 'science'),
    'fiction': train_corpus('fiction.txt', 'fiction')
}

query = "яблоко"
for name, m in models_corpus.items():
    if query in m.wv:
        print(f"{name}: {m.wv.most_similar(query, topn=5)}")
    else:
        print(f"{name}: слово отсутствует")

**7. Детекция семантических сдвигов (Задание 8)**

In [ ]:
# Загружаем корпуса 1995 и 2025 годов (условно)
sentences_1995 = ...   # после предобработки
sentences_2025 = ...
model_1995 = Word2Vec(sentences_1995, vector_size=100, window=5, min_count=2)
model_2025 = Word2Vec(sentences_2025, vector_size=100, window=5, min_count=2)

def semantic_shift(word, m1, m2):
    try:
        v1 = m1.wv[word] / np.linalg.norm(m1.wv[word])
        v2 = m2.wv[word] / np.linalg.norm(m2.wv[word])
        return 1 - np.dot(v1, v2)   # расстояние (0 – без сдвига, 2 – максимально)
    except:
        return None

words_to_check = ["сеть", "облако", "социальный"]
for w in words_to_check:
    shift = semantic_shift(w, model_1995, model_2025)
    if shift is not None:
        print(f"{w}: сдвиг = {shift:.4f}")
        print("  1995 ближайшие:", model_1995.wv.most_similar(w, topn=3))
        print("  2025 ближайшие:", model_2025.wv.most_similar(w, topn=3))

**8. Упрощённая реализация Skip-gram (Задание 9)**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import random

# Строим словарь по предложениям
word_counts = Counter()
for sent in sentences:
    word_counts.update(sent)

vocab = [word for word, cnt in word_counts.items() if cnt >= 2]
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}
vocab_size = len(vocab)

data = [[word2idx[w] for w in sent if w in word2idx] for sent in sentences]

# Создаём пары (центр, контекст) для окна 2
window = 2
pairs = []
for sent in data:
    for i, center in enumerate(sent):
        left = max(0, i - window)
        right = min(len(sent), i + window + 1)
        for j in range(left, right):
            if j != i:
                pairs.append((center, sent[j]))

# Распределение для негативной выборки
freqs = np.array([word_counts[word] for word in vocab], dtype=np.float32)
freqs = freqs ** 0.75
freqs = freqs / freqs.sum()

embed_dim = 100
neg_samples = 5

class SkipGramNeg(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.center_emb = nn.Embedding(vocab_size, embed_dim)
        self.context_emb = nn.Embedding(vocab_size, embed_dim)
        self.log_sigmoid = nn.LogSigmoid()
    
    def forward(self, center, pos_context, neg_context):
        center_vec = self.center_emb(center)
        pos_vec = self.context_emb(pos_context)
        neg_vec = self.context_emb(neg_context)
        
        pos_score = torch.sum(center_vec * pos_vec, dim=1)
        pos_loss = -self.log_sigmoid(pos_score).mean()
        
        neg_score = torch.bmm(neg_vec, center_vec.unsqueeze(2)).squeeze(2)
        neg_loss = -self.log_sigmoid(-neg_score).mean()
        
        return pos_loss + neg_loss

model_scratch = SkipGramNeg(vocab_size, embed_dim)
optimizer = optim.Adam(model_scratch.parameters(), lr=0.01)

epochs = 5
for epoch in range(epochs):
    total_loss = 0
    random.shuffle(pairs)
    for center, pos in pairs:
        negs = np.random.choice(vocab_size, size=neg_samples, p=freqs)
        center_t = torch.tensor([center], dtype=torch.long)
        pos_t = torch.tensor([pos], dtype=torch.long)
        neg_t = torch.tensor(negs, dtype=torch.long).unsqueeze(0)
        
        optimizer.zero_grad()
        loss = model_scratch(center_t, pos_t, neg_t)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, loss = {total_loss / len(pairs):.4f}")

# После обучения можно использовать эмбеддинги:
embeddings_scratch = model_scratch.center_emb.weight.data.numpy()